### RAG Foundations

#### What problem does RAG solve?
Imagine you ask an LLM:

***What is the leave policy of my company?***

The model doesn't know.

Why?

Because your company handbook wasn't part of its training.

Without RAG:

```
User Question
↓
LLM
↓
"I don't know"
```
or worse:
```
User Question
↓
LLM
↓
Hallucination
```
RAG solves this.

RAG =
```
Retrieval
Augmented
Generation
```
Meaning:
```
Find relevant information
+
Give it to the LLM
+
Generate answer
```

### High-Level Pipeline
This is the diagram you should memorize:
```
Documents
↓
Chunking
↓
Embeddings
↓
Vector DB
↓
Retrieval
↓
Augmentation
↓
LLM
↓
Answer
```

### Documents
Documents are simply your knowledge source.

Examples:
```
PDFs

Word Documents

Web Pages

Database Records

FAQs

Support Articles
```
Example:
```
Employee Handbook.pdf
```
contains:
```
Leave Policy

Work From Home Policy

Holiday Policy
```

### Chunking
Problem

Suppose:
```
Employee Handbook

100 pages
```
Embedding the entire PDF as one vector is a bad idea.

Why?

Because one embedding would represent:
```
Leave Policy
+
WFH Policy
+
Benefits
+
Payroll
+
Everything
```
Too much information mixed together.

Solution

Split the document.
```
PDF
↓
Chunking
↓
Smaller Pieces
```
Example:
```
Chunk 1
Company Introduction

Chunk 2
Leave Policy

Chunk 3
Work From Home

Chunk 4
Benefits
```
Now each chunk has a focused meaning.

Breaking large information into manageable pieces.

### Chunk Overlap

Splitting a document raises a hidden problem: **what if an idea sits exactly on the boundary between two chunks?**

#### The problem

Suppose the text is:

```
Employees get 20 annual leave days.
Unused leave can be carried forward to next year.
```

If we cut right in the middle:

```
Chunk 1:
"Employees get 20 annual leave days."

Chunk 2:
"Unused leave can be carried forward to next year."
```

Now a user asks:

```
How many leave days can I carry forward?
```

The answer needs **both** sentences, but they live in **different chunks**. Retrieval might grab only one, and the LLM gives an incomplete answer.

#### The fix: overlap

We let each chunk **repeat a little of the previous chunk's text**.

```
Chunk 1:
"Employees get 20 annual leave days.
 Unused leave can be carried forward..."

Chunk 2:
"...Unused leave can be carried forward
 to next year."
```

The shared text is the **overlap**.

```
[----- Chunk 1 -----]
              [----- Chunk 2 -----]
              ^^^^^^^^
              overlap
```

#### Mental model

```
chunk size     →  how big each piece is
chunk overlap  →  how much neighbors share
```

```
No overlap   →  risk of cutting an idea in half
Some overlap →  ideas survive across chunk boundaries
```

A common starting point is a small overlap (for example, 10–20% of the chunk size). Too little risks losing context; too much wastes storage and adds duplicate text.

```
Goal: every important idea stays whole
      inside at least one chunk.
```

### Embedding the Chunks
Now each chunk becomes a vector.

Example:
```
Chunk:
Employees get 20 annual leave days.

↓

Embedding Model

↓

[0.82, 0.45, ...]
```
Store:
```
Chunk Text
+
Embedding
```

### Vector Database
Store all chunk embeddings.

Example:
```
Chunk 1
Embedding A

Chunk 2
Embedding B

Chunk 3
Embedding C
```
inside:

* Qdrant
* Pinecone
* Chroma
* Weaviate

Purpose:
```
Store vectors

Search vectors

Return similar vectors
```

### Retrieval
Now a user asks:
```
How many annual leave days do employees get?
```
Convert question to embedding.
```
Question
↓
Embedding
↓
Vector
```
Compare against stored chunks.

Example:
```
Chunk 1
0.12

Chunk 2
0.96

Chunk 3
0.25
```
Highest score:
```
Chunk 2
```
This chunk talks about leave policy.

Perfect.

This step is called:
Retrieval
```
Question
↓
Find Relevant Chunks
```

### Top-K Retrieval
Maybe one chunk isn't enough.

Let's retrieve:
```
Top 3 Chunks
```
Example:
```
Leave Policy
0.96

Leave Approval Process
0.91

Leave Carry Forward
0.89
```
Return all three.

This gives more context.

Pipeline:
```
Question
↓
Similarity Search
↓
Top-K Retrieval
↓
Relevant Chunks
```

### Augmentation
This is where beginners get confused.

Retrieval and Augmentation are different.

Retrieval means:

```
Find Information
```
Augmentation means:
```
Add Information To Prompt
```
Example:

User asks:
```
How many annual leave days do employees get?
```
Retrieved chunk:
```
Employees receive 20 annual leave days.
```
Now create a new prompt:
```
Answer using the context below.

Context:
Employees receive 20 annual leave days.

Question:
How many annual leave days do employees get?
```
Notice:
```
Question
+
Retrieved Chunk
```
This combination is called:
**Augmentation**

### Generation
Now the LLM finally gets involved.

Input:
```
Question
+
Retrieved Context
↓
LLM

↓
Employees receive 20 annual leave days.
```
This step is called:
**Generation**
Because the LLM generates the final response.

### Putting It All Together
Imagine:
```
User:
How many annual leave days do employees get?
```
Step 1:
```
Employee Handbook.pdf
↓
Chunking
```
Step 2:
```
Chunks
↓
Embeddings
```
Step 3:
```
Store in Vector DB
```
Step 4:
```
Question
↓
Embedding
```
Step 5:
```
Similarity Search
↓
Top-K Chunks
```
Step 6:
```
Question
+
Retrieved Chunks
```
This is:

**Augmentation**

Step 7:
```
Send to LLM
↓
Generate Answer
```
This is:

**Generation**

This is the entire RAG foundation:
```
Documents
↓
Chunking
↓
Embeddings
↓
Vector DB
↓
Question
↓
Question Embedding
↓
Similarity Search
↓
Top-K Retrieval
↓
Retrieved Chunks
↓
Augmentation
↓
LLM
↓
Generation
↓
Answer
```